# T2. [연습] v1.1 임베딩 + FAISS + 검색 평가

`T1_v11_preprocess (test).ipynb`가 만든 `corpus.jsonl`, `eval.jsonl`로 검색을 해 봅니다.

**이 노트북이 하는 일**
1. **임베딩**: 문단 하나하나를 숫자 768개짜리 벡터로 바꿉니다. 뜻이 비슷한 글은 비슷한 벡터가 됩니다.
2. **FAISS 인덱스**: 벡터들을 빠르게 검색할 수 있게 저장합니다.
3. **검색 평가**: 질문을 넣어서 정답 문단을 상위 몇 위 안에 찾아오는지 채점하고, 단어 일치 방식(BM25)과 비교합니다.

**런타임: GPU (A100 또는 T4).** 위에서부터 순서대로 실행하세요. 마지막 서버 종료 셀은 Run All로는 실행되지 않게 막아 두었습니다.

## 1. 설치 + 저장 위치

In [1]:
!pip -q install sentence-transformers faiss-cpu bm25s

In [2]:
# 드라이브 연결 (안 되면 서버 안에 저장 → 서버를 끄면 사라짐)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/rag_msmarco/v11_test'
except Exception as e:
    import os
    if os.path.exists('/content'):            # Colab인데 드라이브만 안 됨
        print('드라이브 연결 실패 → 서버 안에 저장합니다:', e)
        SAVE_DIR = '/content/rag_msmarco/v11_test'
    else:                                     # 내 컴퓨터에서 실행
        print('Colab이 아님 → 노트북 옆 폴더에 저장합니다')
        SAVE_DIR = os.path.abspath('v11_test_data')
print('저장 위치:', SAVE_DIR)

Mounted at /content/drive
저장 위치: /content/drive/MyDrive/rag_msmarco/v11_test


## 2. 설정 + 파일 불러오기

In [3]:
import os, re, json, random, time
from collections import Counter
import numpy as np

def save_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]

EMB_MODEL    = 'BAAI/bge-base-en-v1.5'     # 영어 임베딩 모델 (768차원)
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '   # bge는 질문에만 이 접두어를 붙임

corpus   = load_jsonl(f'{SAVE_DIR}/corpus.jsonl')
eval_set = load_jsonl(f'{SAVE_DIR}/eval.jsonl')
print('문단', f'{len(corpus):,}', '/ 평가 질문', len(eval_set))

문단 142,605 / 평가 질문 550


## 3. 임베딩

- 문단은 그대로, 질문에는 접두어를 붙여서 넣습니다.
- `normalize_embeddings=True`로 벡터 길이를 1로 맞추면, 두 벡터의 **내적 = 코사인 유사도**가 됩니다.
- 한 번 만든 임베딩은 파일로 저장하고, 다시 실행하면 불러옵니다.

In [4]:
import torch
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(EMB_MODEL, device='cuda' if torch.cuda.is_available() else 'cpu')
if model.device.type == 'cuda':
    model.half()                   # GPU에서 절반 정밀도 → 2배 정도 빠름
model.max_seq_length = 256
print('장치:', model.device)

EMB_PATH = f'{SAVE_DIR}/corpus_emb_bge.npy'
if os.path.exists(EMB_PATH) and len(np.load(EMB_PATH, mmap_mode='r')) == len(corpus):
    emb = np.load(EMB_PATH)
    print('저장된 임베딩 불러옴')
else:
    t = time.time()
    emb = model.encode([c['text'] for c in corpus], batch_size=512, normalize_embeddings=True,
                       convert_to_numpy=True, show_progress_bar=True).astype('float16')
    np.save(EMB_PATH, emb)
    print(f'임베딩 {time.time() - t:.0f}초')
print('모양:', emb.shape, '(문단 수, 차원)')

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

장치: cuda:0


Batches:   0%|          | 0/279 [00:00<?, ?it/s]

임베딩 59초
모양: (142605, 768) (문단 수, 차원)


### 임베딩 감 잡기

뜻이 비슷한 문장끼리 유사도가 높게 나오는지 직접 확인해 봅니다.

In [5]:
sents = ['What are the symptoms of the flu?',
         'Influenza usually causes fever, cough and body aches.',
         'The stock market fell sharply today.']
v = model.encode(sents, normalize_embeddings=True)
print('질문 ↔ 독감 문장 유사도:', round(float(v[0] @ v[1]), 3))
print('질문 ↔ 주식 문장 유사도:', round(float(v[0] @ v[2]), 3))

질문 ↔ 독감 문장 유사도: 0.775
질문 ↔ 주식 문장 유사도: 0.396


## 4. FAISS 인덱스

`IndexFlatIP`는 질문 벡터와 모든 문단 벡터의 내적을 전부 계산해서 큰 순서로 돌려줍니다.
문단 15만 개 정도면 이 방식으로도 충분히 빠르고, 결과가 정확합니다.

In [6]:
import faiss

index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb.astype('float32'))
faiss.write_index(index, f'{SAVE_DIR}/faiss_bge_flatip.index')
print('인덱스 문단 수:', index.ntotal)

인덱스 문단 수: 142605


## 5. 검색 평가 (BM25 vs 임베딩)

건강 질문 500개로 잽니다.

- **Recall@k**: 상위 k개 안에 정답 문단이 하나라도 있는 질문의 비율
- **MRR@10**: 첫 정답 문단 순위의 역수 평균. 1위면 1, 2위면 0.5, 10위 밖이면 0 (MS MARCO 공식 지표)

In [7]:
def retrieval_metrics(ranked, qs, ks=(1, 5, 10, 100)):
    res = {f'Recall@{k}': 0.0 for k in ks}
    res['MRR@10'] = 0.0
    for pids, q in zip(ranked, qs):
        gold = set(q['gold_pids'])
        for k in ks:
            res[f'Recall@{k}'] += bool(gold & set(pids[:k]))
        for r, p in enumerate(pids[:10]):
            if p in gold:
                res['MRR@10'] += 1 / (r + 1)
                break
    return {k: round(v / len(qs), 4) for k, v in res.items()}

eval_ans = [q for q in eval_set if q['answerable']]
queries  = [q['query'] for q in eval_ans]
print('채점할 질문:', len(eval_ans))

채점할 질문: 500


In [8]:
# 임베딩 검색
q_emb = model.encode([QUERY_PREFIX + q for q in queries], batch_size=128,
                     normalize_embeddings=True, convert_to_numpy=True)
_, dense_ids = index.search(q_emb.astype('float32'), 100)
dense_scores = retrieval_metrics(dense_ids.tolist(), eval_ans)
print('임베딩(bge):', dense_scores)

임베딩(bge): {'Recall@1': 0.236, 'Recall@5': 0.596, 'Recall@10': 0.754, 'Recall@100': 0.976, 'MRR@10': 0.385}


In [9]:
# BM25 검색 (단어가 겹치는 정도로 찾는 전통 방식 → 비교 기준선)
import bm25s

bm25 = bm25s.BM25()
bm25.index(bm25s.tokenize([c['text'] for c in corpus], stopwords='en'))
bm25_ids, _ = bm25.retrieve(bm25s.tokenize(queries, stopwords='en'), k=100)
bm25_scores = retrieval_metrics(bm25_ids.tolist(), eval_ans)
print('BM25:', bm25_scores)

Split strings:   0%|          | 0/142605 [00:00<?, ?it/s]

DEBUG:bm25s:Building index from IDs objects


BM25S Count Tokens:   0%|          | 0/142605 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/142605 [00:00<?, ?it/s]

Split strings:   0%|          | 0/500 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/500 [00:00<?, ?it/s]

BM25: {'Recall@1': 0.118, 'Recall@5': 0.34, 'Recall@10': 0.466, 'Recall@100': 0.798, 'MRR@10': 0.213}


In [10]:
# 결과 표 + 질문 종류별 Recall@10 + 저장
import pandas as pd

table = pd.DataFrame({'BM25': bm25_scores, 'bge-base': dense_scores}).T
display(table)

by_type = {}
for name, ids in [('BM25', bm25_ids.tolist()), ('bge-base', dense_ids.tolist())]:
    for t_ in sorted({q['query_type'] for q in eval_ans}):
        sel = [(p, q) for p, q in zip(ids, eval_ans) if q['query_type'] == t_]
        by_type.setdefault(t_, {'n': len(sel)})[name] = retrieval_metrics([p for p, _ in sel], [q for _, q in sel])['Recall@10']
display(pd.DataFrame(by_type).T)

with open(f'{SAVE_DIR}/retrieval_results.json', 'w') as f:
    json.dump({'n_queries': len(eval_ans), 'n_passages': len(corpus),
               'bm25': bm25_scores, 'bge-base': dense_scores, 'recall@10_by_type': by_type}, f, indent=2)

,Recall@1,Recall@5,Recall@10,Recall@100,MRR@10
BM25,0.118,0.340,0.466,0.798,0.213
bge-base,0.236,0.596,0.754,0.976,0.385


,n,BM25,bge-base
description,313.0,0.4984,0.7955
entity,92.0,0.4565,0.7391
location,11.0,0.4545,0.7273
numeric,82.0,0.3537,0.6098
person,2.0,0.5000,1.0000


In [11]:
# 예시: 질문 하나의 검색 결과 상위 3개 (✅ = 정답 문단)
q = eval_ans[0]
print('질문:', q['query'], '\n정답:', q['answers'][0], '\n')
for name, ids in [('BM25', bm25_ids[0]), ('bge-base', dense_ids[0])]:
    print(f'[{name}]')
    for p in ids[:3]:
        mark = '✅' if p in q['gold_pids'] else '  '
        print(f' {mark} [{p}] {corpus[p]["text"][:160]}')

질문: What are the symptoms and causes of conjunctivitis 
정답: Symptoms are Redness in the white of the eye or inner eyelid,Increased amount of tears,Thick yellow discharge that crusts over the eyelashes, especial 

[BM25]
    [117942] Bacteria can be eradicated with antibiotic eye drops or ointments as well as tablets, capsules and liquids. A virus cannot. Unfortunately, viral conjunctivitis 
    [799] Viral conjunctivitis and bacterial conjunctivitis may affect one or both eyes. Viral conjunctivitis usually produces a watery discharge. Bacterial conjunctiviti
    [50360] All forms of conjunctivitis including bacterial, viral, allergic and other types involve inflammation of the transparent, mucous membrane (conjunctiva) covering
[bge-base]
    [112068] Causes and symptoms. Conjunctivitis may be caused by a viral infection, such as a cold, acute respiratory infection, or disease such as measles, herpes simplex,
    [800] Symptoms of conjunctivitis. The symptoms of conjunctivitis will dep

## 6. 결과 요약

In [12]:
EXPECTED = ['corpus.jsonl', 'eval.jsonl', 'meta.json', 'corpus_emb_bge.npy',
            'faiss_bge_flatip.index', 'retrieval_results.json']
missing = [n for n in EXPECTED if not os.path.exists(f'{SAVE_DIR}/{n}')]
for n in EXPECTED:
    if n not in missing:
        print(f'  ✅ {n:26s} {os.path.getsize(f"{SAVE_DIR}/{n}") / 1e6:9.1f} MB')
print('빠진 파일:', missing or '없음')
print('저장 위치:', SAVE_DIR, '(드라이브)' if SAVE_DIR.startswith('/content/drive') else '(서버 안 - 서버를 끄면 사라짐!)')

def to_md(df):
    try:
        return df.to_markdown()
    except ImportError:
        return '```\n' + df.to_string() + '\n```'

summary = '\n'.join([
    f"# [연습 v1.1] 검색 결과 요약 ({time.strftime('%Y-%m-%d %H:%M')})", '',
    f"- 문단 모음: {len(corpus):,}개 {dict(Counter(c['src'] for c in corpus))}",
    f"- 평가: 건강 질문 {len(eval_ans)}개로 검색 채점", '',
    to_md(table), '', '### 질문 종류별 Recall@10', '', to_md(pd.DataFrame(by_type).T),
])
with open(f'{SAVE_DIR}/results_summary.md', 'w', encoding='utf-8') as f:
    f.write(summary)
print('\n' + summary)

  ✅ corpus.jsonl                    78.9 MB
  ✅ eval.jsonl                       0.2 MB
  ✅ meta.json                        0.0 MB
  ✅ corpus_emb_bge.npy             219.0 MB
  ✅ faiss_bge_flatip.index         438.1 MB
  ✅ retrieval_results.json           0.0 MB
빠진 파일: 없음
저장 위치: /content/drive/MyDrive/rag_msmarco/v11_test (드라이브)

# [연습 v1.1] 검색 결과 요약 (2026-10-29 02:53)

- 문단 모음: 142,605개 {'dev_health': 14024, 'train_health': 104494, 'dev_other': 24087}
- 평가: 건강 질문 500개로 검색 채점

|          |   Recall@1 |   Recall@5 |   Recall@10 |   Recall@100 |   MRR@10 |
|:---------|-----------:|-----------:|------------:|-------------:|---------:|
| BM25     |      0.118 |      0.34  |       0.466 |        0.798 |    0.213 |
| bge-base |      0.236 |      0.596 |       0.754 |        0.976 |    0.385 |

### 질문 종류별 Recall@10

|             |   n |   BM25 |   bge-base |
|:------------|----:|-------:|-----------:|
| description | 313 | 0.4984 |     0.7955 |
| entity      |  92 | 0.4565 |     0.7391 |
| 

## 7. 드라이브 저장 마무리 + 런타임 종료

결과 파일은 앞 셀에서 드라이브(`{SAVE_DIR}/`)에 저장됩니다. 이 셀은 저장을 마무리하고 Colab 런타임을 끕니다.

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()   # 결과 파일 드라이브 저장 마무리
runtime.unassign()          # 런타임 종료


In [14]:
shutdown = True